## CLASSIFICATION PART B : 
# Here we will be covering algorithms 
6 Random Forest Classifier	feature importance; n_estimators \
  7	AdaBoost Classifier	n_estimators, learning_rate \
  8	Gradient Boosting Classifier	learning rate (sklearn GBM) \
  9	Bagging Classifier	Decision Tree as base estimator \
  10 MLP Classifier (Neural Network)

In [255]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,StratifiedGroupKFold
import os


In [256]:
folder = 'data/raw/toniot'

path = os.path.join('..',folder,'train_test_network.csv')

df = pd.read_csv(path)

df

,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,-,290.371539,101568,2592,OTH,...,0,0,-,-,-,-,-,-,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000102,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000148,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
3,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000113,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
4,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000130,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211038,192.168.1.32,48286,176.28.50.165,80,tcp,http,65.376610,2665,322,S3,...,0,0,-,-,-,-,-,-,1,xss
211039,192.168.1.32,48288,176.28.50.165,80,tcp,http,65.710346,1987,322,S3,...,0,0,-,-,-,-,-,-,1,xss
211040,192.168.1.32,48292,176.28.50.165,80,tcp,http,65.766512,3922,322,S3,...,0,0,-,-,-,-,-,-,1,xss
211041,192.168.1.32,48294,176.28.50.165,80,tcp,http,65.753940,2401,322,S3,...,0,0,-,-,-,-,-,-,1,xss


In [257]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211043 entries, 0 to 211042
Data columns (total 44 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   src_ip                  211043 non-null  object 
 1   src_port                211043 non-null  int64  
 2   dst_ip                  211043 non-null  object 
 3   dst_port                211043 non-null  int64  
 4   proto                   211043 non-null  object 
 5   service                 211043 non-null  object 
 6   duration                211043 non-null  float64
 7   src_bytes               211043 non-null  int64  
 8   dst_bytes               211043 non-null  int64  
 9   conn_state              211043 non-null  object 
 10  missed_bytes            211043 non-null  int64  
 11  src_pkts                211043 non-null  int64  
 12  src_ip_bytes            211043 non-null  int64  
 13  dst_pkts                211043 non-null  int64  
 14  dst_ip_bytes        

In [258]:
df.isna().sum()
df['http_user_agent'].value_counts()

http_user_agent
-                                                                                                                                    210756
Microsoft-Delivery-Optimization/10.0                                                                                                     62
Microsoft-Windows/10.0 UPnP/1.0                                                                                                          47
Ruby                                                                                                                                     31
DAFUPnP                                                                                                                                  27
User-Agent: Microsoft-DLNA DLNADOC/1.50                                                                                                  24
Windows-Update-Agent/10.0.10011.16384 Client-Protocol/1.91                                                                               21
Micr

In [259]:
df = df.drop_duplicates(ignore_index=True)
df

,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,-,290.371539,101568,2592,OTH,...,0,0,-,-,-,-,-,-,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000102,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000148,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
3,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000113,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
4,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000130,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190469,192.168.1.32,48286,176.28.50.165,80,tcp,http,65.376610,2665,322,S3,...,0,0,-,-,-,-,-,-,1,xss
190470,192.168.1.32,48288,176.28.50.165,80,tcp,http,65.710346,1987,322,S3,...,0,0,-,-,-,-,-,-,1,xss
190471,192.168.1.32,48292,176.28.50.165,80,tcp,http,65.766512,3922,322,S3,...,0,0,-,-,-,-,-,-,1,xss
190472,192.168.1.32,48294,176.28.50.165,80,tcp,http,65.753940,2401,322,S3,...,0,0,-,-,-,-,-,-,1,xss


# log1p on duration, src_bytes, dst_bytes, src_pkts, dst_pkts 
### Are to be log1p transformed as their values are so far apart for some network flows 
### means it is right skewed and then use standard scalar over the columns
### Collapse 5 DNS booleans → 1 dns_present (16.5%, healthy)

## Here we Are StratifiedGroupKFold as it allows when 2 groups are present instead of dividing its values among sets it makes sure whole groups data stays in either of the sets 
### In case of cancer patients
### • The model is forced to learn actual features of cancer, not just recognize specific patients.


In [260]:
NEED = ["proto","service","conn_state","duration","src_bytes","dst_bytes","missed_bytes",
        "src_pkts","dst_pkts","src_ip_bytes","dst_ip_bytes","dns_qtype","dns_rcode",
        "dns_AA","dns_RD","dns_RA","dns_rejected","dns_qclass","dst_port","src_ip","label","type"]
LOG  = ["duration","src_bytes","dst_bytes","src_pkts","dst_pkts","src_ip_bytes","dst_ip_bytes"]
CAT  = ["proto","service","conn_state"]
DNS_S   = ["dns_AA","dns_RD","dns_RA","dns_rejected"]
DROP_AFT = DNS_S + ["dns_qclass", "missed_bytes", "src_ip"]

cols = [
    'http_user_agent',
    'http_orig_mime_types',
    'http_resp_mime_types',
    'weird_name',
    'weird_addl',
    'weird_notice'
]
cols1 = [
    'http_trans_depth',
    'http_method',
    'http_uri',
    'http_version'
]


ssl_keep = [
    "ssl_version",
    "ssl_cipher",
    "ssl_resumed",
    "ssl_established"
]

ssl_drop = ["ssl_subject", "ssl_issuer"]

df[ssl_keep] = df[ssl_keep].replace("-", "Unknown")
df = df.drop(columns=ssl_drop)

df=df.drop(cols+cols1, axis=1)


df = df.sample(frac=1,random_state=42,ignore_index=True)

train,test = next(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42).split(df,df['type'],groups=df['src_ip']))

# here as StratifiedGroupKFold splits and produces 5 divisions out of these the next() chooses
# first split
X_train = df.iloc[train]
X_test = df.iloc[test]

fit = {
  "dur_hi":X_train["duration"].quantile(0.999),
  "ports": set(X_train["dst_port"].value_counts().head(10)),
  "cats":{c:pd.Categorical(X_train[c]).categories for c in CAT}
}

def build(d,fit):
  df = d.copy()
  df['dns_present'] = d[DNS_S].ne('-').any(axis=1).astype("int8")
  df['dns_query_parsed'] = d['service'].eq("dns").astype("int8")
  df['has_missed'] = df['missed_bytes'].gt(0).astype("int8")
  df['duration'] = df['duration'].clip(upper=fit["dur_hi"]) # ,d['dst_port]
  df[LOG] = np.log1p(df[LOG].to_numpy()) # we are applying log transformations
  df["dst_port"] = np.where(df['dst_port'].isin(fit["ports"]),df['dst_port'],-1).astype("int16")
  for c in CAT:
    df[c] =pd.Categorical(df[c],categories=fit['cats'][c]).codes.astype("int16")
  return df.drop(columns=DROP_AFT+["type","label"])


Y_train,Y_test = df["type"].iloc[train].values,df["type"].iloc[test].values

X_train,X_test = build(X_train,fit), build(df.iloc[test],fit)

X_train

# DROP 18 COLUMNS AS THEIR INCLUSION SPOILS THE CLUSTERING TO A HIGH VALUE 
# LEADING TO INCOSISTENT PREDICTIONS

# here dst_port can be 80,98,443 and -1(if not in top10) 
# writing .cat.codes makes 80 encoded as 1 and 98 as 2 and 443 as 3 and -1 as 4

# missed_bytes column correspond to when both direction s of a TCP conversations and send payload to each other . if some packets go missing the bytes inside those missing packets can't be recovered which are recorded as missed_bytes

C:\Users\chand\AppData\Local\Temp\ipykernel_7928\1795924185.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[ssl_keep] = df[ssl_keep].replace("-", "Unknown")


,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,src_pkts,...,ssl_version,ssl_cipher,ssl_resumed,ssl_established,http_request_body_len,http_response_body_len,http_status_code,dns_present,dns_query_parsed,has_missed
1,49989,192.168.1.37,-1,1,0,0.000095,0.000000,0.000000,1,0.693147,...,Unknown,Unknown,Unknown,Unknown,0,0,0,0,0,0
3,48624,192.168.1.190,-1,1,5,0.002208,5.049856,6.480045,10,1.791759,...,Unknown,Unknown,Unknown,Unknown,0,0,0,0,0,0
4,37180,52.28.231.150,-1,1,0,0.000000,0.000000,0.000000,11,0.693147,...,Unknown,Unknown,Unknown,Unknown,0,0,0,0,0,0
6,60526,192.168.1.152,-1,1,0,0.001271,0.000000,0.000000,7,1.098612,...,Unknown,Unknown,Unknown,Unknown,0,0,0,0,0,0
7,1553,192.168.1.195,-1,1,0,0.246755,0.000000,0.000000,6,4.795791,...,Unknown,Unknown,Unknown,Unknown,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190466,54524,192.168.1.152,-1,1,5,0.001914,5.036953,6.139885,10,1.791759,...,Unknown,Unknown,Unknown,Unknown,0,0,0,0,0,0
190468,1226,192.168.1.152,-1,1,0,0.210555,0.000000,0.000000,6,4.795791,...,Unknown,Unknown,Unknown,Unknown,0,0,0,0,0,0
190469,52156,192.168.1.1,-1,1,5,0.006497,5.257495,6.899723,10,1.791759,...,Unknown,Unknown,Unknown,Unknown,0,0,0,0,0,0
190471,33292,192.168.1.195,-1,1,5,0.010453,5.049856,6.886532,10,1.791759,...,Unknown,Unknown,Unknown,Unknown,0,0,0,0,0,0


In [261]:
from sklearn.ensemble import (
  RandomForestClassifier,
  AdaBoostClassifier,
  GradientBoostingClassifier,
  BaggingClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer

numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_test.select_dtypes(include='object').columns.tolist()


numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=20
    ))
])


processor = ColumnTransformer([
    ("num", numeric_pipe, numerical_cols),
    ("cat", categorical_pipe, categorical_cols)
])

X_train1 = processor.fit_transform(X_train)
X_test1 = processor.transform(X_test)

feature_names = processor.get_feature_names_out()

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(include="object").columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=20
    ))
])

processor = ColumnTransformer([
    ("num", numeric_pipe, numerical_cols),
    ("cat", categorical_pipe, categorical_cols)
])

X_train1 = processor.fit_transform(X_train)
X_test1 = processor.transform(X_test)

# Names for BOTH numerical and categorical features
feature_names = processor.get_feature_names_out()

X_train_df = pd.DataFrame(
    X_train1.toarray() if hasattr(X_train1, "toarray") else X_train1,
    columns=feature_names,
    index=X_train.index
)

X_test_df = pd.DataFrame(
    X_test1.toarray() if hasattr(X_test1, "toarray") else X_test1,
    columns=feature_names,
    index=X_test.index
)

print("Train shape:", X_train_df.shape)
print("Test shape:", X_test_df.shape)

X_train_df

Train shape: (123473, 165)
Test shape: (67001, 165)


,num__src_port,num__dst_port,num__proto,num__service,num__duration,num__src_bytes,num__dst_bytes,num__conn_state,num__src_pkts,num__src_ip_bytes,...,cat__ssl_cipher_TLS_AES_128_GCM_SHA256,cat__ssl_cipher_TLS_ECDHE_RSA_WITH_AES_128_GCM_SHA256,cat__ssl_cipher_Unknown,cat__ssl_cipher_infrequent_sklearn,cat__ssl_resumed_F,cat__ssl_resumed_T,cat__ssl_resumed_Unknown,cat__ssl_established_F,cat__ssl_established_T,cat__ssl_established_Unknown
1,0.464720,-0.011384,-0.591837,-0.759393,-0.357891,-0.795590,-0.646065,-1.309432,-0.512292,-0.497642,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,0.374406,-0.011384,-0.591837,1.774126,-0.355598,1.049942,1.341354,1.022999,0.508282,0.609391,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
4,-0.382774,-0.011384,-0.591837,-0.759393,-0.357994,-0.795590,-0.646065,1.282158,-0.512292,-0.457385,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
6,1.161890,-0.011384,-0.591837,-0.759393,-0.356614,-0.795590,-0.646065,0.245522,-0.135628,-0.068987,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
7,-2.739998,-0.011384,-0.591837,-0.759393,-0.090142,-0.795590,-0.646065,-0.013637,3.298927,1.854390,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190466,0.764774,-0.011384,-0.591837,1.774126,-0.355916,1.045227,1.237028,1.022999,0.508282,0.606966,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
190468,-2.761634,-0.011384,-0.591837,-0.759393,-0.129437,-0.795590,-0.646065,-0.013637,3.298927,1.854390,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
190469,0.608097,-0.011384,-0.591837,1.774126,-0.350942,1.125827,1.470069,1.022999,0.508282,0.584603,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
190471,-0.640020,-0.011384,-0.591837,1.774126,-0.346647,1.049942,1.466023,1.022999,0.508282,0.547756,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


In [262]:
  # RandomForestClassifier,
  # AdaBoostClassifier,
  # GradientBoostingClassifier,
  # BaggingClassifier,
# MLPClassifier
models = {
  "RandomForest": RandomForestClassifier(n_estimators=100,random_state=42,n_jobs=-1),
  "AdaBoost": AdaBoostClassifier(n_estimators=100,learning_rate=0.1,random_state=42),
  "Gradient Boost": GradientBoostingClassifier(n_estimators=100,max_depth=4,random_state=42),
  "Bagging": BaggingClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
  "MLP":MLPClassifier(hidden_layer_sizes=(100,),
                      activation="relu",solver="adam",max_iter=300,random_state=42)
}


In [ ]:
from sklearn.metrics import accuracy_score,f1_score

def cls_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1 weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
results=[]


# Store results keyed by model name so re-running a cell UPDATES its row
# instead of appending a duplicate (fixes identical repeated rows).
_model_rows = {}


def add_result(name, m, split="test"):
    _model_rows[name] = {
        "Model": name,
        "Accuracy " + split: m["Accuracy"],
        "F1w " + split: m["F1 weighted"],
    }
    return _model_rows


def results_table():
    import pandas as pd
    return pd.DataFrame(list(_model_rows.values()))


## 6.RandomForest CLASSIFIER

In [264]:
name = "Random Forest"
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

X_train_1, Y_train_1 = (X_train_df, Y_train)
X_test_1 = X_test_df

model.fit(X_train_1, Y_train)
m_test = cls_metrics(Y_test, model.predict(X_test_1))
add_result(name, m_test)

print(results_table().to_string(index=False))

        Model  Accuracy test  F1w test  F1m test
Random Forest       0.666766  0.605872  0.406468


## 7.ADABoost Classifier

In [265]:
from sklearn.tree import DecisionTreeClassifier

name = "AdaBoost"
# Default base estimator is a 1-node decision STUMP which is far too weak for
# 10 imbalanced classes (it only predicted ~5 classes, ~10% acc). Using a
# depth-3 tree as the weak learner lifts accuracy to ~62% (all 10 classes).
model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=3, random_state=42),
    n_estimators=100,
    learning_rate=1.0,
    random_state=42,
)

X_train_1, Y_train_1 = (X_train_df, Y_train)
X_test_1 = X_test_df

model.fit(X_train_1, Y_train)
m_test = cls_metrics(Y_test, model.predict(X_test_1))
add_result(name, m_test)

print(results_table().to_string(index=False))

        Model  Accuracy test  F1w test  F1m test
Random Forest       0.666766  0.605872  0.406468
     AdaBoost       0.641095  0.665019  0.451984


## 8.GradientBoosting Classifier

In [266]:
name="GradientBoostingClassifier"

model = GradientBoostingClassifier(n_estimators=100,learning_rate=0.1,max_depth=3,random_state=42)

X_train_1, Y_train_1 = (X_train_df, Y_train)
X_test_1 = X_test_df

model.fit(X_train_1, Y_train)
m_test = cls_metrics(Y_test, model.predict(X_test_1))
add_result(name, m_test)

print(results_table().to_string(index=False))



                     Model  Accuracy test  F1w test  F1m test
             Random Forest       0.666766  0.605872  0.406468
                  AdaBoost       0.641095  0.665019  0.451984
GradientBoostingClassifier       0.757839  0.752968  0.520674


## 9.Bagging Classifier

In [273]:
name = "Bagging"

# BUG FIX: was `max_samples=1` (int = 1 sample/tree -> 1-node stump, ~9.8% acc,
# predicted only 1 class). Correct: max_samples=1.0 (fraction = 100% bootstrap).
# max_features=0.3 subsamples features per split (Random-Forest-style), which
# decorrelates the trees and lifts accuracy to ~72% (and runs ~5x faster).
model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    max_samples=1.0,
    max_features=0.3,
    bootstrap=True,
    random_state=42,
    n_jobs=-1,
)

X_train_1, Y_train_1 = (X_train_df, Y_train)
X_test_1 = X_test_df

model.fit(X_train_1, Y_train)
m_test = cls_metrics(Y_test, model.predict(X_test_1))
add_result(name, m_test)

print(results_table().to_string(index=False))

                     Model  Accuracy test  F1w test  F1m test
             Random Forest       0.666766  0.605872  0.406468
                  AdaBoost       0.641095  0.665019  0.451984
GradientBoostingClassifier       0.757839  0.752968  0.520674
         BaggingClassifier       0.653393  0.659871  0.448062
                   Bagging       0.726735  0.692078  0.462044


## 10.MLP Classifier

In [ ]:
name = "Neural Network"

# BUG FIX: early_stopping=True crashed with
#   TypeError: ufunc 'isnan' not supported for the input types
# because pandas 3.0 stores the string target as an ArrowStringArray and
# sklearn 1.8 runs np.isnan() on the string predictions during early-stopping
# validation scoring. Removing early_stopping avoids that code path and trains
# fine on the string labels (acc ~0.66, all 10 classes).
model = MLPClassifier(
        hidden_layer_sizes=(100, 50),
        activation="relu",
        solver="adam",
        alpha=0.0001,
        learning_rate_init=0.001,
        max_iter=300,
        random_state=42
    )


X_train_1, Y_train_1 = (X_train_df, Y_train)
X_test_1 = X_test_df

model.fit(X_train_1, Y_train)
m_test = cls_metrics(Y_test, model.predict(X_test_1))
add_result(name, m_test)

print(results_table().to_string(index=False))

                     Model  Accuracy test  F1w test  F1m test
             Random Forest       0.666766  0.605872  0.406468
                  AdaBoost       0.641095  0.665019  0.451984
GradientBoostingClassifier       0.757839  0.752968  0.520674
         BaggingClassifier       0.653393  0.659871  0.448062
                   Bagging       0.726735  0.692078  0.462044
            Neural Network       0.660289  0.621252  0.405462


: 